# 07 - Generate statistics CSVs
Creating a separate notebook to calculate the statistics all in one pass so that I can complete a representative set of CSVs for each instrument class.

### Statistics for QARTOD tests in production

In [1]:
# Import libraries available from main conda channels or conda-forge
import xarray as xr
import pandas as pd
import numpy as np
import glob
import os
import re
import warnings
warnings.filterwarnings("ignore")

# Import dask tools and ProgressBar
import dask
from dask.diagnostics import ProgressBar

# Import qartod_testing project functions
from qartod_testing.qc_flag_statistics import ooinet_gold_copy_request, \
    get_test_parameters, parse_qartod_executed, qartod_summary_expanded, \
    get_deployment_ds, collect_statistics

# Import OOI library functions
from ooi_data_explorations.common import merge_frames
from ooinet.M2M import get_deployments, get_annotations

In [2]:
# Setup parameters needed to request data
refdes = "GP03FLMB-RIM01-02-CTDMOG064"
method = "recovered_inst"
stream = "ctdmo_ghqr_instrument_recovered"

# Site, node, and sensor info from deconstructed reference designator
# [site, node, sensor] = refdes.split('-', 2)

In [3]:
# Routine in data_processing module from this project to download the gold copy THREDDs datasets
# Variable 'files' contains list of catalog URLs for downloaded datasets 
files = ooinet_gold_copy_request(refdes, method, stream)

In [4]:
# Load expected results data from external data folder
folder_path = os.path.join(os.path.abspath('../data/external'), method, stream, refdes)
expected_files = glob.glob(folder_path+'/*.nc')
expected_files.sort() # sorts local test files in alphanumeric order

In [5]:
expected_files
# get_deployments(refdes)

['/home/jovyan/code/qartod_testing/data/external/recovered_inst/ctdmo_ghqr_instrument_recovered/GP03FLMB-RIM01-02-CTDMOG064/deployment0001_GP03FLMB-RIM01-02-CTDMOG064-recovered_inst-ctdmo_ghqr_instrument_recovered_20130724T064501-20140618T134501.nc',
 '/home/jovyan/code/qartod_testing/data/external/recovered_inst/ctdmo_ghqr_instrument_recovered/GP03FLMB-RIM01-02-CTDMOG064/deployment0002_GP03FLMB-RIM01-02-CTDMOG064-recovered_inst-ctdmo_ghqr_instrument_recovered_20140620T033001-20150606T013001.nc',
 '/home/jovyan/code/qartod_testing/data/external/recovered_inst/ctdmo_ghqr_instrument_recovered/GP03FLMB-RIM01-02-CTDMOG064/deployment0003_GP03FLMB-RIM01-02-CTDMOG064-recovered_inst-ctdmo_ghqr_instrument_recovered_20150608T213001-20160703T143001.nc',
 '/home/jovyan/code/qartod_testing/data/external/recovered_inst/ctdmo_ghqr_instrument_recovered/GP03FLMB-RIM01-02-CTDMOG064/deployment0004_GP03FLMB-RIM01-02-CTDMOG064-recovered_inst-ctdmo_ghqr_instrument_recovered_20160704T231501-20170717T150001.n

In [6]:
get_annotations(refdes)

,@class,id,subsite,node,sensor,method,stream,beginDT,endDT,annotation,exclusionFlag,source,qcFlag,parameters
0,.AnnotationRecord,154,GP03FLMB,NaN,NaN,None,None,1374647520000,1403049600000,Deployment 1: Mooring was deployed 5-10m shall...,False,lgarzio@marine.rutgers.edu,0,[]
1,.AnnotationRecord,156,GP03FLMB,NaN,NaN,None,None,1433798700000,1467570900000,Deployment 3: Mooring was deployed 10m deeper ...,False,lgarzio@marine.rutgers.edu,0,[]
2,.AnnotationRecord,157,GP03FLMB,NaN,NaN,None,None,1467673620000,1500303720000,Deployment 4: Mooring was deployed 10m deeper ...,False,lgarzio@marine.rutgers.edu,0,[]
3,.AnnotationRecord,155,GP03FLMB,NaN,NaN,None,None,1403229600000,1433635200000,Deployment 2: Mooring was deployed 10-15m shal...,False,lgarzio@marine.rutgers.edu,0,[]
4,.AnnotationRecord,142,GP03FLMB,NaN,NaN,None,None,1500072540000,1532538660000,Deployment 5: Telemetered data are not availab...,False,lgarzio@marine.rutgers.edu,0,[]
5,.AnnotationRecord,2169,GP03FLMB,RIM01,02-CTDMOG064,None,None,1500072540000,1532538660000,Deployment 5: It was discovered upon recovery ...,False,cdobson@whoi.edu,9,[]
6,.AnnotationRecord,4757,GP03FLMB,NaN,NaN,None,None,1684525200000,1717688760000,Deployment 10: * UPDATED 2024-01-23: Telemeter...,False,cdobson@whoi.edu,0,[]


#### Gross range test statistics

In [7]:
gross_range_stats = collect_statistics(expected_files, "gross_range")
gross_range_stats

Evaluating statistics on QARTOD flags for deployment             01
Evaluating statistics on QARTOD flags for deployment             02
Evaluating statistics on QARTOD flags for deployment             03
Evaluating statistics on QARTOD flags for deployment             04
Evaluating statistics on QARTOD flags for deployment             05
Evaluating statistics on QARTOD flags for deployment             06
Evaluating statistics on QARTOD flags for deployment             07
Evaluating statistics on QARTOD flags for deployment             08
Evaluating statistics on QARTOD flags for deployment             09
Evaluating statistics on QARTOD flags for deployment             10
Evaluating statistics on QARTOD flags for deployment             11
Evaluating statistics on QARTOD flags for all deployments


,sea_water_electrical_conductivity total,conductivity good,conductivity good %,conductivity suspect,conductivity suspect %,conductivity fail,conductivity fail %,sea_water_pressure total,pressure good,pressure good %,...,temperature suspect %,temperature fail,temperature fail %,sea_water_practical_salinity total,salinity good,salinity good %,salinity suspect,salinity suspect %,salinity fail,salinity fail %
deployment,,,,,,,,,,,,,,,,,,,,,
01,31613,31612,100.00,1,0.00,0,0.0,31613,31611,99.99,...,0.01,0,0.0,31613,31613,100.00,0,0.00,0,0.0
02,33689,33619,99.79,70,0.21,0,0.0,33689,33686,99.99,...,0.01,0,0.0,33689,33618,99.79,71,0.21,0,0.0
03,37509,37508,100.00,1,0.00,0,0.0,37509,37505,99.99,...,0.01,0,0.0,37509,37509,100.00,0,0.00,0,0.0
04,36256,36255,100.00,1,0.00,0,0.0,36256,36254,99.99,...,0.05,0,0.0,36256,36255,100.00,1,0.00,0,0.0
05,36065,36065,100.00,0,0.00,0,0.0,36065,36061,99.99,...,0.00,0,0.0,36065,36065,100.00,0,0.00,0,0.0
06,41283,41282,100.00,1,0.00,0,0.0,41283,41280,99.99,...,0.00,0,0.0,41283,41251,99.92,32,0.08,0,0.0
07,64210,64210,100.00,0,0.00,0,0.0,64210,64207,100.00,...,0.17,0,0.0,64210,64210,100.00,0,0.00,0,0.0
08,28981,28981,100.00,0,0.00,0,0.0,28981,28977,99.99,...,0.00,0,0.0,28981,28981,100.00,0,0.00,0,0.0
09,35412,35412,100.00,0,0.00,0,0.0,35412,35408,99.99,...,0.00,0,0.0,35412,35410,99.99,2,0.01,0,0.0


In [8]:
# Save data frames as CSVs
folder_path = os.path.join(os.path.abspath('../data/processed'), method, stream, refdes)
os.makedirs(folder_path, exist_ok=True)
gross_range_stats.to_csv(folder_path+f"/gross_range-{refdes}-flag_statistics.csv", na_rep='NaN', mode='w')

#### Climatology test statistics

In [9]:
climatology_stats = collect_statistics(expected_files, "climatology")
climatology_stats

Evaluating statistics on QARTOD flags for deployment             01
Evaluating statistics on QARTOD flags for deployment             02
Evaluating statistics on QARTOD flags for deployment             03
Evaluating statistics on QARTOD flags for deployment             04
Evaluating statistics on QARTOD flags for deployment             05
Evaluating statistics on QARTOD flags for deployment             06
Evaluating statistics on QARTOD flags for deployment             07
Evaluating statistics on QARTOD flags for deployment             08
Evaluating statistics on QARTOD flags for deployment             09
Evaluating statistics on QARTOD flags for deployment             10
Evaluating statistics on QARTOD flags for deployment             11
Evaluating statistics on QARTOD flags for all deployments


,sea_water_electrical_conductivity total,sea_water_pressure total,sea_water_temperature total,temperature good,temperature good %,temperature suspect,temperature suspect %,temperature fail,temperature fail %,sea_water_practical_salinity total,salinity good,salinity good %,salinity suspect,salinity suspect %,salinity fail,salinity fail %
deployment,,,,,,,,,,,,,,,,
01,NaN,NaN,31613,31611,99.99,2,0.01,0,0.0,31613,31602,99.97,11,0.03,0,0.0
02,NaN,NaN,33689,33685,99.99,4,0.01,0,0.0,33689,33617,99.79,72,0.21,0,0.0
03,NaN,NaN,37509,37507,99.99,2,0.01,0,0.0,37509,37507,99.99,2,0.01,0,0.0
04,NaN,NaN,36256,36161,99.74,95,0.26,0,0.0,36256,36256,100.00,0,0.00,0,0.0
05,NaN,NaN,36065,36064,100.00,1,0.00,0,0.0,36065,36064,100.00,1,0.00,0,0.0
06,NaN,NaN,41283,41265,99.96,18,0.04,0,0.0,41283,41250,99.92,33,0.08,0,0.0
07,NaN,NaN,64210,63785,99.34,425,0.66,0,0.0,64210,62636,97.55,1574,2.45,0,0.0
08,NaN,NaN,28981,28981,100.00,0,0.00,0,0.0,28981,28981,100.00,0,0.00,0,0.0
09,NaN,NaN,35412,35412,100.00,0,0.00,0,0.0,35412,35318,99.73,94,0.27,0,0.0


In [10]:
# Save data frames as CSVs
folder_path = os.path.join(os.path.abspath('../data/processed'), method, stream, refdes)
os.makedirs(folder_path, exist_ok=True)
climatology_stats.to_csv(folder_path+f"/climatology-{refdes}-flag_statistics.csv", na_rep='NaN', mode='w')